In [2]:
import argparse
from typing import Optional
import time
from tqdm import tqdm
import numpy as np
import pickle
import networkx as nx
import math
import multiprocessing as mp
from functools import partial
import os

In [3]:
route_path = "C:/Users/jecla/Documents/Barcelona_GNN/data/processed/Linköping/routing_cache/kshortest_paths.pkl"

In [4]:
# ==========================================
# 1. CARGA Y EXPLORACIÓN DEL GRAFO
# ==========================================

def load_and_explore_graph(path):
    """
    Carga un grafo serializado desde disco y muestra información básica.

    Args:
        path (str): Ruta al archivo pickle que contiene el grafo.

    Returns:
        networkx.Graph or DiGraph: El grafo cargado, o None si el archivo no existe.

    Efectos secundarios:
        - Imprime por consola el número de nodos/edges, tipo de grafo y ejemplos de atributos
          en nodos para inspección rápida.

    Uso:
        graph = load_and_explore_graph('path/to/graph.pkl')
    """
    # Verificar si el archivo existe
    if not os.path.exists(path):
        print(f"Error: El archivo no existe en {path}")
        return None

    # Cargar con pickle (se espera un grafo NetworkX serializado)
    with open(path, 'rb') as f:
        graph = pickle.load(f)

    # Mensajes informativos básicos
    print(f"Grafo cargado: {graph.number_of_nodes()} nodos, {graph.number_of_edges()} aristas")
    print(f"Tipo de grafo: {type(graph)}")

    # Mostrar ejemplos de nodos y sus atributos (hasta 4 ejemplos)
    print("\nEjemplo de nodos:")
    for i, (node_id, data) in enumerate(graph.nodes(data=True)):
        print(f"Node ID: {node_id}, Data: {data}")
        if i > 2: break

    # Mostrar qué atributos están definidos en los nodos (si existen)
    print("\nCaracterísticas disponibles en nodos:")
    if graph.nodes:
        print(list(graph.nodes(data=True))[0][1].keys())

    return graph

# %%
# %%
# ==========================================
# 2. PREPARACIÓN DE DATOS (NODOS Y PARES OD)
# ==========================================

def prepare_od_pairs(graph, limit=200):
    """
    Genera una lista de pares origen-destino (OD) a partir de nodos de tipo 'taz' y 'aux'.

    Args:
        graph (networkx.Graph): Grafo con atributos de nodos que incluyen 'type'.
        limit (int): Número máximo de pares OD a generar.

    Returns:
        list of tuple: Lista de tuplas (origin, destination) limitada por 'limit'.

    Notas:
        - Se priorizan nodos con atributo 'type' == 'taz' y 'aux'.
        - Evita pares donde origen == destino.
        - Útil para crear casos de prueba rápidos en benchmarking.
    """
    # Contar tipos de nodos para entender la composición del grafo
    node_type_counts = {}
    for _, data in graph.nodes(data=True):
        node_type = data.get('type', 'unknown')
        node_type_counts[node_type] = node_type_counts.get(node_type, 0) + 1

    print("\nConteo de nodos por tipo:", node_type_counts)

    # Filtrar nodos por tipo: TAZs y nodos auxiliares
    taz_nodes = [n for n, d in graph.nodes(data=True) if d.get('type') == 'taz']
    aux_nodes = [n for n, d in graph.nodes(data=True) if d.get('type') == 'aux']
    combined_nodes = taz_nodes + aux_nodes

    # Generador de pares OD (limitado o ilimitado)
    # Si limit es None o negativo, generamos todos los pares posibles (excepto origen==destino)
    od_pairs = []
    count = 0
    unlimited = (limit is None) or (isinstance(limit, int) and limit < 0)
    for origin in combined_nodes:
        for destination in combined_nodes:
            if origin == destination:
                continue
            od_pairs.append((origin, destination))
            count += 1
            if not unlimited and count >= limit:
                break
        if not unlimited and count >= limit:
            break

    print(f"Total de pares OD generados: {len(od_pairs)}")
    return od_pairs

In [7]:
# Fix for argparse in Colab/Jupyter environments
parser = argparse.ArgumentParser(description='Benchmark de algoritmos de enrutamiento')

if '__file__' not in locals():
    # Running in an interactive environment like Colab/Jupyter
    args = parser.parse_args([])
else:
    # Running as a standalone script
    args = parser.parse_args()

# 1. Cargar Grafo
# (Si falla la ruta, intenta pasarla explícitamente al ejecutar el script)
if not os.path.exists(args.graph):
    # Fallback para pruebas si no existe la ruta hardcodeada
    print(f"Advertencia: No se encontró {args.graph}. Asegúrate de pasar la ruta correcta.")
    graph = None
else:
    graph = load_and_explore_graph(args.graph)

if graph:
    # 2. Preparar Pares OD
    od_pairs = prepare_od_pairs(graph, limit=args.limit if args.limit >= 0 else None)

AttributeError: 'Namespace' object has no attribute 'graph'

In [ ]:
missing_nodes_in_od_pairs = set()

# Obtener todos los nodos presentes en el grafo para una búsqueda eficiente
graph_nodes = set(graph.nodes())

for origin, destination in od_pairs:
    if origin not in graph_nodes:
        missing_nodes_in_od_pairs.add(origin)
    if destination not in graph_nodes:
        missing_nodes_in_od_pairs.add(destination)

if missing_nodes_in_od_pairs:
    print(f"Se encontraron {len(missing_nodes_in_od_pairs)} nodos en 'od_pairs' que NO están en el grafo:")
    # Mostrar hasta los primeros 10 nodos faltantes si hay muchos
    for i, node_id in enumerate(list(missing_nodes_in_od_pairs)[:10]):
        print(f" - {node_id}")
    if len(missing_nodes_in_od_pairs) > 10:
        print("   ...")
else:
    print("Todos los nodos en 'od_pairs' existen en el grafo. El problema de rutas vacías no parece ser por nodos inexistentes.")

In [8]:
# Seleccionar el primer par OD para la prueba
test_origin, test_destination = od_pairs[0]
print(f"Probando un par OD individual: {test_origin} -> {test_destination}")

# --- Prueba 1: nx.shortest_path (Dijkstra) ---
print("\nIntentando con nx.shortest_path (Dijkstra)...")
try:
    path_dijkstra = nx.shortest_path(graph, test_origin, test_destination, weight='free_flow_time')
    print(f"  Ruta encontrada por nx.shortest_path (Dijkstra): {path_dijkstra}")
    print(f"  Longitud de la ruta (Dijkstra): {len(path_dijkstra)}")
except nx.NetworkXNoPath:
    print(f"  No se encontró ruta por nx.shortest_path (Dijkstra) para {test_origin} -> {test_destination}")
except Exception as e:
    print(f"  Error inesperado con nx.shortest_path (Dijkstra): {e}")

# --- Prueba 2: nx.astar_path ---
print("\nIntentando con nx.astar_path...")
try:
    # La función heurística ya fue definida y verificada en _yen_k_shortest_astar
    # Recreamos la heurística localmente para esta prueba directa
    def h(u, v):
        try:
            return math.dist(graph.nodes[u]['pos'], graph.nodes[v]['pos'])
        except KeyError:
            # Si por alguna razón falta 'pos' aquí, aunque ya se verificó antes
            return 0

    path_astar = nx.astar_path(graph, test_origin, test_destination, heuristic=h, weight='free_flow_time')
    print(f"  Ruta encontrada por nx.astar_path: {path_astar}")
    print(f"  Longitud de la ruta (A*): {len(path_astar)}")
except nx.NetworkXNoPath:
    print(f"  No se encontró ruta por nx.astar_path para {test_origin} -> {test_destination}")
except Exception as e:
    print(f"  Error inesperado con nx.astar_path: {e}")

NameError: name 'od_pairs' is not defined